In [ ]:
# ---------------------------------------
# Phase 1.2: Data Storage Setup in GCS
# ---------------------------------------

from google.colab import auth
auth.authenticate_user()

import os
from google.cloud import storage

# 🔹 Replace with your GCP project ID & bucket name
PROJECT_ID = "ishikagcp027"
BUCKET_NAME = "technet-data"

# Initialize client
storage_client = storage.Client(project=PROJECT_ID)

# Check if bucket exists, if not create it
try:
    bucket = storage_client.get_bucket(BUCKET_NAME)
    print(f"✅ Bucket {BUCKET_NAME} already exists.")
except Exception as e:
    bucket = storage_client.create_bucket(BUCKET_NAME, location="US")
    print(f"✅ Created bucket {BUCKET_NAME} in US region.")

# Define folder structure inside bucket
folders = ["incidents", "technical"]
for folder in folders:
    blob = bucket.blob(f"{folder}/")
    blob.upload_from_string("")  # creates folder placeholder
    print(f"📂 Created folder gs://{BUCKET_NAME}/{folder}/")

# ---------------------------------------
# Upload cleaned CSVs
# ---------------------------------------
local_files = {
    "cleaned_data/clean_incident_records.csv": "incidents/simple_incident_records.csv",
    "cleaned_data/clean_tech_records.csv": "technical/simple_tech_records.csv",
}

for local_path, gcs_path in local_files.items():
    blob = bucket.blob(gcs_path)
    blob.upload_from_filename(local_path)
    print(f"⬆️ Uploaded {local_path} → gs://{BUCKET_NAME}/{gcs_path}")

# ---------------------------------------
# Configure IAM permissions (basic demo)
# ---------------------------------------
# Example: grant Vertex AI service account read access
# Replace with your actual service account email
service_account = "ishikagcp027@appspot.gserviceaccount.com"

policy = bucket.get_iam_policy(requested_policy_version=3)
policy.bindings.append({
    "role": "roles/storage.objectViewer",
    "members": {f"serviceAccount:{service_account}"}
})
bucket.set_iam_policy(policy)

print(f"✅ Granted read access to {service_account}")

# ---------------------------------------
# Simple Data Validation Pipeline
# ---------------------------------------
import pandas as pd

def validate_csv(local_file, required_columns):
    try:
        df = pd.read_csv(local_file)
        missing_cols = [col for col in required_columns if col not in df.columns]
        if missing_cols:
            print(f"❌ Validation failed: Missing columns {missing_cols} in {local_file}")
            return False
        if df.isnull().sum().any():
            print(f"⚠️ Warning: Missing values found in {local_file}")
        print(f"✅ Validation passed for {local_file}")
        return True
    except Exception as e:
        print(f"❌ Error validating {local_file}: {e}")
        return False

# Validate datasets before upload
validate_csv("cleaned_data/clean_incident_records.csv", ["TicketID", "ProductID", "ProblemDescription"])
validate_csv("cleaned_data/clean_tech_records.csv", ["DocID", "ProductID", "step_description"])


In [ ]:
!pip install --quiet google-cloud-aiplatform vertexai


In [ ]:
!gcloud config set project ishikagcp027
!gcloud services enable aiplatform.googleapis.com
!gcloud services enable generativelanguage.googleapis.com


In [ ]:
!gcloud projects add-iam-policy-binding ishikagcp027 \
  --member="serviceAccount:ishikagcp027@appspot.gserviceaccount.com" \
  --role="roles/aiplatform.user"

!gcloud projects add-iam-policy-binding ishikagcp027 \
  --member="serviceAccount:ishikagcp027@appspot.gserviceaccount.com" \
  --role="roles/storage.objectAdmin"